# 03 · Research-Domain Classification and Keywords

**Use Case 3: Research Paper Analysis and Recommendation System**  
**Owner:** Member B (classification lead)

**Goal:** Predict a paper's primary research domain (6 CS categories) and extract its keywords.

**Basic techniques (Weeks 1–6)**
- TF-IDF keyword extraction (top-weighted terms per paper and per category)
- Traditional classification: Naive Bayes and Logistic Regression on BoW vs TF-IDF features

**Advanced techniques (Weeks 7–12)**
- Fine-tune DistilBERT/SciBERT **or** zero/few-shot LLM classification
- Optional: ensemble of the best traditional model and the transformer

**Evaluation:** Accuracy, macro-F1 and a confusion matrix on the shared test split for every model. Discuss cross-listed papers (about 54% of the sample) as a source of error.

**Report section:** Classification; the comparative-analysis table across all members (~1,000 words).

## Setup

Run `01_data_loading_and_sampling.ipynb` first. Always evaluate on `test_df`.

In [ ]:
from utils import load_sample, load_split

df = load_sample()
train_df, test_df = load_split(df)
print(f"all={len(df):,}  train={len(train_df):,}  test={len(test_df):,}")
train_df[["id", "primary_category", "title"]].head()

In [ ]:
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import transformers
import os
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
transformers.logging.set_verbosity_error()
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.naive_bayes import MultinomialNB

import classify as clf
from extraction import clean_text
from utils import save_results, ARTIFACTS_DIR

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

CATEGORIES = clf.CATEGORIES
y_train = train_df["primary_category"]
y_test = test_df["primary_category"]

# Sizes and models for the slower advanced parts (subsets, as agreed in the team plan)
ZERO_SHOT_N = 500             # W7 zero-shot on 500 test papers
FINETUNE_TRAIN_N = 3000       # W9 DistilBERT fine-tune on 3,000 training papers, 1 epoch
LLM_N = 200                   # W9 few-shot LLM classifier on 200 test papers
ZERO_SHOT_MODEL = "sileod/deberta-v3-base-tasksource-nli"   # same model as the Week 7 practical
DISTILBERT = "distilbert-base-uncased"                      # Week 6 tokenizer and Week 9 fine-tuning model

HAS_GPU = torch.cuda.is_available()
print("GPU available:", HAS_GPU, "|", torch.cuda.get_device_name(0) if HAS_GPU else "CPU only")
results = {"models": []}      # one row per model; saved with save_results("03", ...) at the end

## TF-IDF keywords

Week 4 TF-IDF, fitted on the training papers only (after `clean_text` from Part A removes LaTeX). A domain's keywords are the terms with the highest **average** TF-IDF weight over that domain's papers; a paper's keywords are its own highest-weighted terms. Unigrams and bigrams are used so phrases like "language models" survive.

> **Justification.** TF-IDF (Week 4) gives high weight to words that are frequent in one paper but rare across the corpus, which is close to what a keyword is. Averaging the weights over a category gives each domain a vocabulary we can check by eye (cs.CR: security, attacks, privacy), and the same weighting is the feature set for the classifiers below. Bigrams keep phrases like "language models" together.

In [ ]:
keyword_vectoriser = TfidfVectorizer(stop_words="english", min_df=3, max_df=0.5,
                                     sublinear_tf=True, ngram_range=(1, 2))
# clean_text (from extraction.py) strips LaTeX such as \textbf{...} so it does not show up as a keyword
X_keywords = keyword_vectoriser.fit_transform(train_df["text"].map(clean_text))
terms = keyword_vectoriser.get_feature_names_out()

domain_keywords = {}
for category in CATEGORIES:
    rows = np.where(y_train.values == category)[0]
    mean_weights = np.asarray(X_keywords[rows].mean(axis=0)).ravel()
    best = np.argsort(mean_weights)[::-1][:10]
    domain_keywords[category] = [terms[j] for j in best]

results["domain_keywords"] = domain_keywords
pd.DataFrame(domain_keywords)

In [ ]:
# Keywords of three individual test papers
for _, row in test_df.sample(n=3, random_state=SEED).iterrows():
    weights = keyword_vectoriser.transform([clean_text(row["text"])]).toarray().ravel()
    best = np.argsort(weights)[::-1][:6]
    print(f"[{row['primary_category']}] {row['title']}")
    print("   ->", [terms[j] for j in best if weights[j] > 0], "\n")

## BoW / TF-IDF + NB / LogReg baselines

Week 5 pipeline: Multinomial Naive Bayes on BoW counts and on TF-IDF, Logistic Regression on TF-IDF (same unigram vocabulary, so only the model or the weighting changes), plus Logistic Regression on unigrams + bigrams. Classes are balanced (1,600 train / 400 test each), so **macro-F1** is the main metric next to accuracy.

> **Justification.** Naive Bayes and Logistic Regression (Week 5) train in seconds, need no GPU and can be inspected, so they set the bar every later model has to clear. Each row changes one thing (counts vs TF-IDF, NB vs LogReg, unigrams vs bigrams), so any gain can be traced to that change. The six classes are balanced, which is why accuracy and macro-F1 come out almost identical.

In [ ]:
def evaluate(y_true, y_pred, model_name, n_note=""):
    """Accuracy and macro precision/recall/F1 (adapted from evaluate_model in the Week 5 practical)."""
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=CATEGORIES, average="macro", zero_division=0)   # labels=: an "unparsed" LLM answer never becomes a 7th class
    row = {"model": model_name, "test_papers": len(y_true), "accuracy": round(accuracy, 3),
           "macro_precision": round(precision, 3), "macro_recall": round(recall, 3), "macro_f1": round(f1, 3)}
    if n_note:
        row["note"] = n_note
    return row


def plot_confusion_matrix(y_true, y_pred, model_name):
    """Week 5 heatmap."""
    cm = confusion_matrix(y_true, y_pred, labels=CATEGORIES)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CATEGORIES, yticklabels=CATEGORIES)
    plt.title(f"Confusion matrix - {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()
    return cm

In [ ]:
bow_vectoriser = CountVectorizer(stop_words="english", min_df=2)
tfidf_vectoriser = TfidfVectorizer(stop_words="english", min_df=2, sublinear_tf=True)

X_train_bow = bow_vectoriser.fit_transform(train_df["text"])
X_test_bow = bow_vectoriser.transform(test_df["text"])
X_train_tfidf = tfidf_vectoriser.fit_transform(train_df["text"])
X_test_tfidf = tfidf_vectoriser.transform(test_df["text"])
print("BoW:", X_train_bow.shape, " TF-IDF:", X_train_tfidf.shape)

In [ ]:
classical_rows = []
train_accuracy = {}

nb_bow = MultinomialNB().fit(X_train_bow, y_train)
classical_rows.append(evaluate(y_test, nb_bow.predict(X_test_bow), "Naive Bayes + BoW"))
train_accuracy["Naive Bayes + BoW"] = accuracy_score(y_train, nb_bow.predict(X_train_bow))

nb_tfidf = MultinomialNB().fit(X_train_tfidf, y_train)
classical_rows.append(evaluate(y_test, nb_tfidf.predict(X_test_tfidf), "Naive Bayes + TF-IDF"))
train_accuracy["Naive Bayes + TF-IDF"] = accuracy_score(y_train, nb_tfidf.predict(X_train_tfidf))

lr_tfidf = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_train_tfidf, y_train)
classical_rows.append(evaluate(y_test, lr_tfidf.predict(X_test_tfidf), "LogReg + TF-IDF"))
train_accuracy["LogReg + TF-IDF"] = accuracy_score(y_train, lr_tfidf.predict(X_train_tfidf))

# Final classical model: the pipeline in classify.py (TF-IDF unigrams + bigrams -> LogReg)
lr_pipeline = clf.build_tfidf_logreg(SEED).fit(train_df["text"], y_train)
lr_test_pred = lr_pipeline.predict(test_df["text"])
classical_rows.append(evaluate(y_test, lr_test_pred, "LogReg + TF-IDF (1-2 grams)"))
train_accuracy["LogReg + TF-IDF (1-2 grams)"] = accuracy_score(y_train, lr_pipeline.predict(train_df["text"]))

results["models"].extend(classical_rows)
classical_table = pd.DataFrame(classical_rows).set_index("model")
classical_table["train_accuracy"] = pd.Series(train_accuracy).round(3)
classical_table["train_test_gap"] = (classical_table["train_accuracy"] - classical_table["accuracy"]).round(3)
classical_table

In [ ]:
lr_cm = plot_confusion_matrix(y_test, lr_test_pred, "LogReg + TF-IDF (1-2 grams)")

path = clf.save_classifier(lr_pipeline)
print("Saved", path.relative_to(ARTIFACTS_DIR.parent), "-> NB06 uses classify.predict()")

## Transformer or LLM classifier

Four advanced classifiers, each compared with the classical baseline above:
1. **Zero-shot NLI** (Week 7): no training; the model scores "This text is about {category name}" for each category.
2. **LSTM** (Week 6, lite): the simplified LSTM from the practical with a Hugging Face tokenizer, 2 epochs.
3. **DistilBERT** (Week 9, lite), both adaptation types from the lecture on the same 3,000 training papers: frozen [CLS] features + Logistic Regression, and a full fine-tune with Hugging Face `Trainer` for 1 epoch (GPU only).
4. **Few-shot LLM** (Week 9 prompting) through the team's `llm.generate`.

### Zero-shot NLI classifier (W7)

NLI model from the Week 7 practical (`sileod/deberta-v3-base-tasksource-nli`). Run twice: with the official category names as labels, and with one-line descriptions condensed from the arXiv taxonomy (`classify.CATEGORY_DESCRIPTIONS`).

> **Justification.** Zero-shot NLI (Week 7) needs no labelled data: each category becomes a hypothesis ("This paper is about ...") and the model picks the one it finds most entailed. It tells us what we would get if arXiv had no labels at all. The bare names worked badly because "Machine Learning" is true of almost every abstract, so the second run swaps in short descriptions and changes nothing else.

In [ ]:
import json
from transformers import pipeline

zero_shot = pipeline("zero-shot-classification", model=ZERO_SHOT_MODEL, device=0 if HAS_GPU else -1)
zs_df = test_df.sample(n=ZERO_SHOT_N, random_state=SEED).reset_index(drop=True)


def run_zero_shot(labels, cache_name):
    """Top-scoring label for each of the 500 papers, cached because the CPU run takes ~7 min."""
    label_to_code = {name: code for code, name in labels.items()}
    cache = ARTIFACTS_DIR / cache_name
    if cache.exists():
        print("Loaded cached zero-shot predictions:", cache_name)
        return json.loads(cache.read_text())
    pred = []
    start = time.time()
    for i, text in enumerate(zs_df["text"]):
        # "This paper is about {}." fits research abstracts better than the default "This example is {}."
        result = zero_shot(clf.shorten(text, 1500), list(label_to_code),
                           hypothesis_template="This paper is about {}.", multi_label=False)
        pred.append(label_to_code[result["labels"][0]])
        if (i + 1) % 100 == 0:
            print(f"{i + 1}/{ZERO_SHOT_N} done ({time.time() - start:.0f} s)")
    cache.write_text(json.dumps(pred))
    return pred


zs_pred = run_zero_shot(clf.CATEGORY_NAMES, "03_zero_shot_pred.json")
zs_desc_pred = run_zero_shot(clf.CATEGORY_DESCRIPTIONS, "03_zero_shot_desc_pred.json")
zs_rows = [
    evaluate(zs_df["primary_category"], zs_pred, "Zero-shot NLI, category names", f"{ZERO_SHOT_N}-paper subset"),
    evaluate(zs_df["primary_category"], zs_desc_pred, "Zero-shot NLI, category descriptions", f"{ZERO_SHOT_N}-paper subset"),
]
results["models"].extend(zs_rows)
pd.DataFrame(zs_rows)

### LSTM (W6, lite)

> **Justification.** The Week 6 LSTM is the first model here that reads words in order instead of as a bag. It learns its embeddings from scratch on 9,600 abstracts, so it shows how far a small sequence model gets without pre-training. Only two epochs ("lite"). It mean-pools over real tokens instead of using the practical's last hidden state, which had mostly seen padding.

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer

device = torch.device("cuda" if HAS_GPU else "cpu")
tokenizer = AutoTokenizer.from_pretrained(DISTILBERT)
label2id = {code: i for i, code in enumerate(CATEGORIES)}
id2label = {i: code for code, i in label2id.items()}


def encode(texts, labels, max_length=256):
    """Tokenise with the Hugging Face tokenizer (Week 6, 2.2) and pack ids + labels into a TensorDataset."""
    tokens = tokenizer(list(texts), padding="max_length", truncation=True,
                       max_length=max_length, return_tensors="pt")
    label_ids = torch.tensor([label2id[label] for label in labels])
    return TensorDataset(tokens["input_ids"], label_ids)


lstm_train = encode(train_df["text"], y_train)
lstm_test = encode(test_df["text"], y_test)
print("train:", len(lstm_train), "test:", len(lstm_test))

In [ ]:
class LSTMClassifier(nn.Module):
    """Week 6 simplified LSTM, changed from 1 sigmoid output to 6 class scores."""

    def __init__(self, vocab_size, num_classes, embedding_dim=128, hidden_dim=64, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        lstm_out, _ = self.lstm(embedded)
        # Mean-pool over real tokens only. Abstracts are right-padded to 256, so the final hidden
        # state (used in the practical) has read 100+ PAD tokens and forgets the text -> near-random accuracy.
        mask = (input_ids != tokenizer.pad_token_id).unsqueeze(-1).float()
        pooled = (lstm_out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.classifier(self.dropout(pooled))   # raw scores; CrossEntropyLoss applies softmax


lstm = LSTMClassifier(vocab_size=len(tokenizer), num_classes=len(CATEGORIES)).to(device)
optimizer = optim.Adam(lstm.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
train_loader = DataLoader(lstm_train, batch_size=32, shuffle=True)
test_loader = DataLoader(lstm_test, batch_size=64)

for epoch in range(2):
    lstm.train()
    total_loss, correct, seen = 0.0, 0, 0
    for input_ids, labels in train_loader:
        input_ids, labels = input_ids.to(device), labels.to(device)
        optimizer.zero_grad()
        scores = lstm(input_ids)
        loss = criterion(scores, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (scores.argmax(dim=1) == labels).sum().item()
        seen += labels.size(0)
    print(f"Epoch {epoch + 1}/2: loss = {total_loss / len(train_loader):.4f}, train accuracy = {correct / seen:.3f}")

lstm.eval()
lstm_pred = []
with torch.no_grad():
    for input_ids, labels in test_loader:
        scores = lstm(input_ids.to(device))
        lstm_pred.extend(id2label[i] for i in scores.argmax(dim=1).cpu().tolist())

lstm_row = evaluate(y_test, lstm_pred, "LSTM (2 epochs)")
results["models"].append(lstm_row)
lstm_row

### DistilBERT: feature extraction vs full fine-tune (W9, lite)

> **Justification.** Week 9 covers two ways to adapt a pre-trained model: feature extraction (freeze the body, train a classifier on top) and full fine-tuning (update every weight). Both run on the same 3,000 papers, next to a LogReg trained on those same 3,000, so the three rows differ only in the model. DistilBERT is the small BERT from the practicals; one epoch takes under a minute on a laptop GPU.

In [ ]:
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoModel, Trainer, TrainingArguments

ft_train_df = train_df.groupby("primary_category", group_keys=False).sample(
    n=FINETUNE_TRAIN_N // len(CATEGORIES), random_state=SEED)          # 500 per category = 3,000
distilbert_rows = []
distilbert_probs = None

# The classical pipeline on the same 3,000 papers, so the comparison is about the model, not the data size
lr_3k = clf.build_tfidf_logreg(SEED).fit(ft_train_df["text"], ft_train_df["primary_category"])
distilbert_rows.append(evaluate(y_test, lr_3k.predict(test_df["text"]), f"LogReg + TF-IDF ({len(ft_train_df)} train)"))

# (a) Feature extraction: DistilBERT frozen, its [CLS] vector goes into Logistic Regression
encoder = AutoModel.from_pretrained(DISTILBERT).to(device).eval()


def cls_vectors(texts, batch_size=32):
    vectors = []
    texts = list(texts)
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = tokenizer(texts[i:i + batch_size], padding=True, truncation=True,
                              max_length=256, return_tensors="pt").to(device)
            vectors.append(encoder(**batch).last_hidden_state[:, 0].cpu().numpy())
    return np.vstack(vectors)


frozen_lr = LogisticRegression(max_iter=2000, random_state=SEED).fit(
    cls_vectors(ft_train_df["text"]), ft_train_df["primary_category"])
frozen_pred = frozen_lr.predict(cls_vectors(test_df["text"]))
distilbert_rows.append(evaluate(y_test, frozen_pred, f"DistilBERT frozen + LogReg ({len(ft_train_df)} train)"))
del encoder

# (b) Full fine-tune with Trainer, every weight updated (GPU only; ~3 min on an RTX 4060)
if HAS_GPU:
    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

    def to_dataset(frame):
        data = Dataset.from_pandas(pd.DataFrame({
            "text": frame["text"].tolist(),
            "labels": [label2id[c] for c in frame["primary_category"]]}))
        return data.map(tokenize_function, batched=True)

    tokenized_train = to_dataset(ft_train_df)
    tokenized_test = to_dataset(test_df)

    model = AutoModelForSequenceClassification.from_pretrained(
        DISTILBERT, num_labels=len(CATEGORIES), id2label=id2label, label2id=label2id)
    training_args = TrainingArguments(
        output_dir=str(ARTIFACTS_DIR / "models" / "distilbert_runs"),
        num_train_epochs=1, per_device_train_batch_size=16, per_device_eval_batch_size=64,
        learning_rate=5e-5, weight_decay=0.01, logging_steps=50,
        eval_strategy="no", save_strategy="no", report_to="none", seed=SEED)
    trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_train)

    start = time.time()
    trainer.train()
    print(f"Fine-tuning took {time.time() - start:.0f} s")

    logits = trainer.predict(tokenized_test).predictions
    distilbert_probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    distilbert_pred = [id2label[i] for i in logits.argmax(axis=1)]
    model.save_pretrained(ARTIFACTS_DIR / "models" / "distilbert")
    tokenizer.save_pretrained(ARTIFACTS_DIR / "models" / "distilbert")
    distilbert_rows.append(evaluate(y_test, distilbert_pred, f"DistilBERT fine-tuned (1 epoch, {len(ft_train_df)} train)"))
else:
    print("No GPU: full fine-tune skipped, only the frozen-feature row is reported")

results["models"].extend(distilbert_rows)
pd.DataFrame(distilbert_rows)

### Few-shot LLM classifier (W9 prompting)

The prompt lists the six arXiv category names, then gives one worked example per category taken from the **training** set, then the test paper. The reply is parsed for a category code; an unparseable reply counts as wrong. Same two label variants as zero-shot: official names vs short descriptions.

> **Justification.** Few-shot prompting (Week 9) tests whether a general instruction-tuned LLM, the same Qwen2.5-1.5B the rest of the system uses, can classify from one example per category with no training. The cost is speed: about 1.7 s per paper on the GPU against a fraction of a millisecond for LogReg. Replies are parsed for a category code, and anything unparseable counts as wrong.

In [ ]:
import llm
print("LLM backend:", llm.describe())

examples = []
for category in CATEGORIES:
    example = train_df[y_train == category].sample(n=1, random_state=SEED).iloc[0]
    examples.append((example["text"], category))

llm_df = test_df.sample(n=LLM_N, random_state=SEED).reset_index(drop=True)
print(clf.build_few_shot_prompt(llm_df.loc[0, "text"], examples)[:1500], "...")

In [ ]:
llm_rows = []
for describe in (False, True):
    llm_pred = []
    start = time.time()
    for i, text in enumerate(llm_df["text"]):
        llm_pred.append(clf.llm_classify(text, examples, describe=describe))
        if (i + 1) % 50 == 0:
            print(f"{i + 1}/{LLM_N} done ({time.time() - start:.0f} s)")

    unparsed = sum(1 for p in llm_pred if p is None)
    llm_pred_filled = [p if p is not None else "unparsed" for p in llm_pred]
    name = "Few-shot LLM, " + ("category descriptions" if describe else "category names")
    llm_rows.append(evaluate(llm_df["primary_category"], llm_pred_filled, name, f"{LLM_N}-paper subset, {unparsed} unparsed"))

results["models"].extend(llm_rows)
pd.DataFrame(llm_rows)

## Ensemble (optional)

Soft voting: average the class probabilities of two models and take the most likely class. LogReg + Naive Bayes always runs (both are cheap); LogReg + fine-tuned DistilBERT runs when the GPU fine-tune above produced probabilities.

> **Justification.** Soft voting averages two models' class probabilities, and it only helps when the models make different mistakes. NB and LogReg share the same features, so little gain is expected from that pair; LogReg + fine-tuned DistilBERT pairs a bag-of-words model with a contextual one, which is more likely to disagree usefully. Ensembles are on the spec's advanced list and in the Week 5 practical's list of improvements.

In [ ]:
lr_probs = lr_pipeline.predict_proba(test_df["text"])           # columns follow lr_pipeline.classes_
nb_probs = nb_tfidf.predict_proba(X_test_tfidf)
assert list(lr_pipeline.classes_) == list(nb_tfidf.classes_) == CATEGORIES

ensemble_rows = []
avg = (lr_probs + nb_probs) / 2
ensemble_rows.append(evaluate(y_test, [CATEGORIES[i] for i in avg.argmax(axis=1)], "Ensemble: LogReg + NB"))

if distilbert_probs is not None:
    avg = (lr_probs + distilbert_probs) / 2
    ensemble_rows.append(evaluate(y_test, [CATEGORIES[i] for i in avg.argmax(axis=1)], "Ensemble: LogReg + DistilBERT"))

results["models"].extend(ensemble_rows)
pd.DataFrame(ensemble_rows)

## Comparison table and error analysis

The zero-shot and LLM rows use subsets, so the LogReg baseline is also scored on exactly the same subsets for a fair comparison.

> **Justification.** Zero-shot and few-shot ran on subsets, so LogReg is re-scored on exactly those papers; comparing a 500-paper number with a 2,400-paper one would not be fair. The confusion pairs and the cross-listed split test whether the errors come from the models or from the labels: 1,332 of the 2,400 test papers are listed under more than one category, and only one of them counts as correct.

In [ ]:
lr_by_id = dict(zip(test_df["id"], lr_test_pred))
fair_rows = [
    evaluate(zs_df["primary_category"], [lr_by_id[i] for i in zs_df["id"]], f"LogReg (same {ZERO_SHOT_N} as zero-shot)", f"{ZERO_SHOT_N}-paper subset"),
    evaluate(llm_df["primary_category"], [lr_by_id[i] for i in llm_df["id"]], f"LogReg (same {LLM_N} as LLM)", f"{LLM_N}-paper subset"),
]
results["models"].extend(fair_rows)

comparison = pd.DataFrame(results["models"]).set_index("model")
comparison.sort_values("macro_f1", ascending=False)

In [ ]:
plot_table = comparison.sort_values("macro_f1")
plt.figure(figsize=(9, 5))
plt.barh(plot_table.index, plot_table["macro_f1"], color="steelblue")
plt.axvline(1 / len(CATEGORIES), color="grey", linestyle="--", label="random guess (1/6)")
plt.xlabel("Macro-F1")
plt.title("Research-domain classification: all models")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Which domains get confused with each other? (off-diagonal cells of the LogReg confusion matrix)
pairs = []
for i, actual in enumerate(CATEGORIES):
    for j, predicted in enumerate(CATEGORIES):
        if i != j:
            pairs.append((actual, predicted, int(lr_cm[i, j])))
confused = pd.DataFrame(sorted(pairs, key=lambda p: -p[2])[:6], columns=["actual", "predicted", "papers"])
results["most_confused_pairs"] = confused.to_dict(orient="records")
confused

In [ ]:
# Cross-listing: are errors concentrated in papers listed under more than one category?
errors = test_df.assign(predicted=lr_test_pred)
errors["correct"] = errors["predicted"] == errors["primary_category"]
errors["cross_listed"] = errors["all_categories"].str.len() > 1
# "lenient" = the prediction is one of the paper's categories, even if not the primary one
errors["lenient_correct"] = [p in cats for p, cats in zip(errors["predicted"], errors["all_categories"])]

by_listing = errors.groupby("cross_listed")["correct"].agg(["count", "mean"]).round(3)
by_listing.index = ["single category", "cross-listed"]
results["accuracy_by_listing"] = by_listing["mean"].to_dict()
results["lenient_accuracy_logreg"] = round(float(errors["lenient_correct"].mean()), 3)
print("Strict accuracy :", round(float(errors["correct"].mean()), 3))
print("Lenient accuracy:", results["lenient_accuracy_logreg"], "(prediction is any of the paper's listed categories)")
by_listing

In [ ]:
# Three misclassified papers, to read by hand
for _, row in errors[~errors["correct"]].sample(n=3, random_state=SEED).iterrows():
    print(f"actual {row['primary_category']} | predicted {row['predicted']} | all categories {row['all_categories']}")
    print("  ", row["title"], "\n")

## Findings and limitations

- **Best model: TF-IDF (1–2 grams) + LogReg**, 0.792 accuracy and macro-F1 on all 2,400 test papers. It is the one saved to `classifier.pkl` for NB06. No advanced model beat it: fine-tuned DistilBERT 0.760, LSTM 0.661, and neither ensemble (LogReg + NB ties at 0.792; adding DistilBERT drops it to 0.777, because plain averaging gives the weaker model equal weight).
- **Same data, different model.** On the same 3,000 training papers: LogReg 0.777, full fine-tune 0.760, frozen features 0.717. Full fine-tuning beats feature extraction, as Week 9 predicts, but one epoch isn't enough to beat word counts on a task where topic words carry most of the signal.
- **For zero-shot, label wording mattered more than the model.** Swapping the arXiv names for one-line descriptions took zero-shot from 0.366 to 0.616 and few-shot from 0.415 to 0.485. With the names, 298 of 500 zero-shot answers were cs.LG. Both remain well below LogReg on the same papers (0.814 and 0.825).
- **Many errors are label overlap, not model failure.** The top confusions are cs.AI ↔ cs.LG and cs.AI ↔ cs.CL. Cross-listed papers score 0.763 against 0.828 for single-category ones, and counting any listed category as correct lifts LogReg to 0.875.
- **Limitations:** zero-shot and few-shot ran on 500 and 200 papers; DistilBERT had 1 epoch on 3k papers and the LSTM 2 epochs; only one 1.5B LLM was tried; the descriptions were written after the names failed on the same subset, so 0.616 is a little optimistic; and each paper gets one label although more than half have several.

In [ ]:
path = save_results("03", results)
print("Saved", path.relative_to(ARTIFACTS_DIR.parent))
comparison